In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries loaded successfully!")


✅ Libraries loaded successfully!


In [2]:
print("Loading forecast data...")

store_1 = pd.read_csv(
    'C:/Users/Shivam Gupta/Coding/smart-supply-chain/data/processed/store_1_data.csv'
)
store_1['Date'] = pd.to_datetime(store_1['Date'], format='mixed', dayfirst=True, errors='coerce')
store_1 = store_1.dropna(subset=['Date'])
store_1_open = store_1[store_1['Open'] == 1].copy()

store_5 = pd.read_csv(
    'C:/Users/Shivam Gupta/Coding/smart-supply-chain/data/processed/store_5_data.csv'
)
store_5['Date'] = pd.to_datetime(store_5['Date'], format='mixed', dayfirst=True, errors='coerce')
store_5 = store_5.dropna(subset=['Date'])
store_5_open = store_5[store_5['Open'] == 1].copy()

print("✅ Data loaded")


Loading forecast data...
✅ Data loaded


In [3]:
params = {
    'unit_cost': 50,
    'holding_cost_percent': 0.25,
    'order_cost': 150,
    'service_level': 0.95,
    'lead_time_days': 7,
    'working_days': 365
}

params['holding_cost_per_unit'] = params['unit_cost'] * params['holding_cost_percent']


In [5]:
def calculate_demand_stats(store_data, store_name):
    daily_sales = store_data['Sales'].values
    return {
        'store': store_name,
        'mean_daily_demand': daily_sales.mean(),
        'std_daily_demand': daily_sales.std(),
        'annual_demand': daily_sales.mean() * params['working_days']
    }

stats_store1 = calculate_demand_stats(store_1_open, "Store 1")
stats_store5 = calculate_demand_stats(store_5_open, "Store 5")


In [6]:
def calculate_eoq(annual_demand, order_cost, holding_cost):
    return np.sqrt((2 * annual_demand * order_cost) / holding_cost)

def total_inventory_cost(Q, D, S, H):
    return (D/Q)*S + (Q/2)*H

results_eoq = []

for stats in [stats_store1, stats_store5]:
    eoq = calculate_eoq(
        stats['annual_demand'],
        params['order_cost'],
        params['holding_cost_per_unit']
    )
    results_eoq.append({
        'Store': stats['store'],
        'EOQ': eoq,
        'Orders_Per_Year': stats['annual_demand'] / eoq
    })

eoq_df = pd.DataFrame(results_eoq)


In [7]:
Z = 1.65  # for 95% service level
results_reorder = []

for stats in [stats_store1, stats_store5]:
    safety_stock = Z * stats['std_daily_demand'] * np.sqrt(params['lead_time_days'])
    reorder_point = stats['mean_daily_demand'] * params['lead_time_days'] + safety_stock

    results_reorder.append({
        'Store': stats['store'],
        'Safety_Stock': safety_stock,
        'Reorder_Point': reorder_point
    })

reorder_df = pd.DataFrame(results_reorder)


In [8]:
savings_results = []

for idx, stats in enumerate([stats_store1, stats_store5]):
    naive_Q = stats['mean_daily_demand'] * 7 * 1.2
    naive_cost = total_inventory_cost(
        naive_Q,
        stats['annual_demand'],
        params['order_cost'],
        params['holding_cost_per_unit']
    )

    eoq_Q = results_eoq[idx]['EOQ']
    optimized_cost = total_inventory_cost(
        eoq_Q,
        stats['annual_demand'],
        params['order_cost'],
        params['holding_cost_per_unit']
    )

    savings_results.append({
        'Store': stats['store'],
        'Current_Cost': naive_cost,
        'Optimized_Cost': optimized_cost,
        'Annual_Savings': naive_cost - optimized_cost,
        'Savings_Percent': (naive_cost - optimized_cost) / naive_cost * 100
    })

savings_df = pd.DataFrame(savings_results)
total_savings = savings_df['Annual_Savings'].sum()


In [9]:
summary_df = pd.DataFrame({
    'Store': eoq_df['Store'],
    'EOQ': eoq_df['EOQ'],
    'Orders_Per_Year': eoq_df['Orders_Per_Year'],
    'Safety_Stock': reorder_df['Safety_Stock'],
    'Reorder_Point': reorder_df['Reorder_Point'],
    'Current_Annual_Cost': savings_df['Current_Cost'],
    'Optimized_Annual_Cost': savings_df['Optimized_Cost'],
    'Annual_Savings': savings_df['Annual_Savings'],
    'Savings_Percent': savings_df['Savings_Percent']
})

display(summary_df)


,Store,EOQ,Orders_Per_Year,Safety_Stock,Reorder_Point,Current_Annual_Cost,Optimized_Annual_Cost,Annual_Savings,Savings_Percent
0,Store 1,6456.754698,269.031446,4415.510449,37729.182664,256370.398756,80709.433724,175660.965032,68.518427
1,Store 5,6400.325497,266.680229,7703.395110,40437.318088,252022.279479,80004.068715,172018.210764,68.255160


In [10]:
base_path = 'C:/Users/Shivam Gupta/Coding/smart-supply-chain/results/tables'
os.makedirs(base_path, exist_ok=True)

summary_df.to_csv(f"{base_path}/inventory_optimization_summary.csv", index=False)
eoq_df.to_csv(f"{base_path}/eoq_results.csv", index=False)
reorder_df.to_csv(f"{base_path}/reorder_point_results.csv", index=False)
savings_df.to_csv(f"{base_path}/cost_savings_analysis.csv", index=False)

print("✅ All result tables saved successfully")


✅ All result tables saved successfully


In [11]:
print("="*80)
print("🎉 WEEK 4 COMPLETE: INVENTORY OPTIMIZATION")
print("="*80)
print(f"💰 Total Annual Savings: ${total_savings:,.2f}")
print("🎯 Service Level Maintained: 95%")


🎉 WEEK 4 COMPLETE: INVENTORY OPTIMIZATION
💰 Total Annual Savings: $347,679.18
🎯 Service Level Maintained: 95%
